# Gemini 3 Flash Knowledge Extraction
This notebook extracts full book knowledge into Supabase using Gemini's long context window.

In [1]:
import os
import sys
from io import BytesIO
import json
from google import genai
from dotenv import load_dotenv

# Add project root to path
sys.path.append('..')

from src.supabase_client import get_supabase_client
from src.ingestion.epub_parser import parse_epub
from src.models import ParsedChapter
from src.knowledge.models import (
    KnowledgeBase, ChapterExtraction, CharacterEntity, CharacterEvent, 
    Relationship, RelationshipMoment, WorldFact, ChapterSummary, ChapterRef
)
from src.knowledge.store import load_knowledge, save_knowledge, is_chapter_extracted
from src.knowledge.merger import merge_extraction

# Load environment variables
load_dotenv('../.env')

client_genai = genai.Client(api_key=os.getenv('GOOGLE_API_KEY'))
print('Environment initialized.')

Environment initialized.


In [2]:
USER_ID = '4e94a213-dae5-4762-8bb4-9e86a7776712'
SERIES_ID = 'red-rising'
BOOK_INDEX = 0

client = get_supabase_client()
book_path = f'{USER_ID}/{SERIES_ID}/book_{BOOK_INDEX}.epub'
print(f'Target: {book_path}')

book_data = client.storage.from_('books').download(book_path)
chapters = parse_epub(BytesIO(book_data))
print(f'Loaded {len(chapters)} chapters.')

Target: 4e94a213-dae5-4762-8bb4-9e86a7776712/red-rising/book_0.epub
Loaded 50 chapters.


In [3]:
def build_batch_prompt(chapter_batch):
    """Build a prompt for a batch of chapters."""
    prompt_parts = [
        'You are an expert literary analyst. Extract structured knowledge from the following chapters.',
        'For each chapter, provide: ',
        '1. Characters: Name (canonical), Role, Description (2-3 sentences), and Key Events in this chapter.',
        '2. Relationships: Character A, Character B, Type (ally, rival, family, romance, mentor, enemy, other), and a Moment describing their interaction.',
        '3. World Facts: Category (location, faction, concept, etc.), Name, and Description.',
        '4. Summary: A 200-300 word narrative recap.',
        '\nIMPORTANT: Return ONLY valid JSON that matches this schema exactly:',
        '{\n  "chapters": [\n    {\n      "index": int,\n      "label": string,\n      "characters": [ { "name": string, "role": string, "description": string, "key_events": [string] } ],\n      "relationships": [ { "character_a": string, "character_b": string, "type": string, "moment": string } ],\n      "world_facts": [ { "category": string, "name": string, "description": string } ],\n      "summary": string\n    }\n  ]\n}'
    ]
    
    for ch in chapter_batch:
        prompt_parts.append(f'<chapter index="{ch.index}" title="{ch.label}">\n{ch.text}\n</chapter>')
    
    return '\n\n'.join(prompt_parts)

In [6]:
import time
import re

def clean_json_response(text):
    """Extract JSON from a response that might contain markdown blocks or extra text."""
    # Find the first { and last }
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        return match.group(0)
    return text

async def run_full_extraction(user_id, series_id, book_index, chapters, batch_size=5, max_retries=3):
    kb = await load_knowledge(series_id, user_id)
    to_extract = [ch for ch in chapters if not is_chapter_extracted(kb, book_index, ch.index)]
    print(f"Remaining chapters to extract: {len(to_extract)}")
    
    for i in range(0, len(to_extract), batch_size):
        batch = to_extract[i : i + batch_size]
        print(f"\nProcessing batch {i//batch_size + 1}: chapters {[ch.index for ch in batch]}...")
        
        prompt = build_batch_prompt(batch)
        
        attempt = 0
        success = False
        while attempt < max_retries and not success:
            try:
                response = client_genai.models.generate_content(
                    model='gemini-flash-latest',
                    contents=prompt,
                    config={'response_mime_type': 'application/json'}
                )
                
                json_text = clean_json_response(response.text)
                data = json.loads(json_text)
                
                for ch_data in data.get('chapters', []):
                    idx = ch_data['index']
                    label = ch_data.get('label', f'Chapter {idx}')
                    ref = ChapterRef(book_index=book_index, chapter_index=idx)
                    
                    chars = [CharacterEntity(
                        name=c['name'],
                        role=c.get('role'),
                        description=c.get('description', ''),
                        first_appearance=ref,
                        key_events=[CharacterEvent(description=ev, book_index=book_index, chapter_index=idx) for ev in c.get('key_events', [])]
                    ) for c in ch_data.get('characters', [])]
                    
                    rels = [Relationship(
                        character_a=r['character_a'],
                        character_b=r['character_b'],
                        type=r.get('type', 'other'),
                        description=r.get('moment', ''),
                        moments=[RelationshipMoment(description=r.get('moment', ''), book_index=book_index, chapter_index=idx)]
                    ) for r in ch_data.get('relationships', [])]
                    
                    facts = [WorldFact(
                        category=f.get('category', 'concept'),
                        name=f['name'],
                        description=f.get('description', ''),
                        book_index=book_index,
                        chapter_index=idx
                    ) for f in ch_data.get('world_facts', [])]
                    
                    summary = ChapterSummary(
                        book_index=book_index,
                        chapter_index=idx,
                        chapter_label=label,
                        summary=ch_data.get('summary', ''),
                        characters_present=[c.name for c in chars]
                    )
                    
                    kb = merge_extraction(kb, ChapterExtraction(characters=chars, relationships=rels, world_facts=facts, summary=summary), book_index, idx)
                
                await save_knowledge(kb, user_id)
                print(f"  → Saved. KB size: {len(kb.characters)} characters.")
                success = True
                
            except Exception as e:
                attempt += 1
                if '503' in str(e) or 'demand' in str(e).lower():
                    wait_time = attempt * 10
                    print(f"  → Server busy (503). Retrying in {wait_time}s... (Attempt {attempt}/{max_retries})")
                    time.sleep(wait_time)
                else:
                    print(f"  → Error: {e}. Retrying... (Attempt {attempt}/{max_retries})")
                    time.sleep(2)

print("Robust extraction functions ready.")

Robust extraction functions ready.


In [7]:
await run_full_extraction(USER_ID, SERIES_ID, BOOK_INDEX, chapters, batch_size=8)

Remaining chapters to extract: 13

Processing batch 1: chapters [32, 33, 34, 35, 36, 37, 38, 39]...
  → Error: Extra data: line 438 column 1 (char 26444). Retrying... (Attempt 1/3)
  → Saved. KB size: 51 characters.

Processing batch 2: chapters [45, 46, 47, 48, 49]...
  → Saved. KB size: 53 characters.
